# Academic Stress & Clinical Referral Ecosystem: Deep Transformer Fine-Tuning
**Student Researcher:** Sahil Mina (24UCOMP047) | Central University of Karnataka
**Guide:** Prof. Srikantaiah K C

This notebook fine-tunes `sentence-transformers/all-MiniLM-L6-v2` on 6,000 balanced real-world reflections across 3 stress tiers:
- **Tier 0:** Low Stress (Eustress / Routine)
- **Tier 1:** Medium Stress (Situational Academic Stress / Exams / Deadlines)
- **Tier 2:** High Stress (Clinical Crisis / Severe Distress Escalation)

After training, it exports the model to **ONNX INT8**, ready to drop into your local project for sub-25ms CPU inference.

In [ ]:
# 1. Install Dependencies
!pip install -q transformers datasets accelerate onnx onnxruntime scikit-learn

In [ ]:
# 2. Upload train.csv, val.csv, and test.csv from your local 'data/' folder
from google.colab import files
print("Please upload your 'train.csv', 'val.csv', and 'test.csv' from your local data/ folder:")
uploaded = files.upload()

In [ ]:
# 3. Load Datasets and Compute Class Weights
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "custom_minilm_stress"

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

print(f"Loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Compute class weights
labels = train_df["label"].values
weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
print("Class weights:", weights)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], max_length=128, padding="max_length", truncation=True)

train_ds = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

In [ ]:
# 4. Define Weighted Loss Trainer and Initialize Model
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fn(logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "Low", 1: "Medium", 2: "High"},
    label2id={"Low": 0, "Medium": 1, "High": 2}
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

# Launch training (Takes ~90 seconds on Colab GPU)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
# 5. Held-out Test Set Evaluation & Edge-Case Verification
test_preds = np.argmax(trainer.predict(test_ds).predictions, axis=1)
print("=" * 55)
print(classification_report(test_df["label"], test_preds, target_names=["Low", "Medium", "High"]))
print("=" * 55)

edge_cases = [
    "tomorrow is exam i'm anxious",
    "viva tomorrow feeling panicky and tense",
    "i want to end my life cannot take this agonizing pain anymore",
    "finished my computer networks lab report feeling great",
    "three lab submissions in two days feeling overwhelmed"
]

model.eval()
model.to("cpu")
tier_names = ["Low", "Medium", "High"]
for t in edge_cases:
    inp = tokenizer(t, return_tensors="pt", max_length=128, padding="max_length", truncation=True)
    probs = torch.softmax(model(**inp).logits[0], dim=0).detach().numpy()
    pred = int(np.argmax(probs))
    print(f"'{t}' -> {tier_names[pred]} | Probs: Low={probs[0]*100:.1f}%, Med={probs[1]*100:.1f}%, High={probs[2]*100:.1f}%")

In [ ]:
# 6. Export to ONNX & Quantize to INT8 for Client-Side CPU Deployment
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

dummy = tokenizer("dummy text", return_tensors="pt", max_length=128, padding="max_length", truncation=True)

torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    "stress_model_fp32.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14
)

quantize_dynamic(
    model_input="stress_model_fp32.onnx",
    model_output="stress_model_quantized.onnx",
    weight_type=QuantType.QInt8
)
print("ONNX INT8 Quantized model created successfully!")

In [ ]:
# 7. Download Model Weights & ONNX Artifacts
import shutil
shutil.make_archive("custom_minilm_stress_weights", "zip", OUTPUT_DIR)
files.download("stress_model_quantized.onnx")
files.download("custom_minilm_stress_weights.zip")
print("Downloaded! Place 'stress_model_quantized.onnx' in your project's 'models/' folder.")